### Query Script Pull-Out

In [2]:
from IPython.display import display, HTML
from sqlalchemy import create_engine, text
import pandas as pd
from dotenv import load_dotenv
import psycopg2
import os

# config

load_dotenv()

USERNAME = os.getenv("DB_USERNAME")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")
QUERY = "query_repo2.sql"

# make engine

url_engine = f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(url_engine)

# read sql file

with open(QUERY, "r", encoding="utf-8") as file:
    query = file.read()

# execute query and convertion

with engine.connect() as conn:
    result = conn.execute(text(query))
    
    # convert to dataframe
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# ultra ready dataframe
# display(df.head(20))

### Quick Check

In [ ]:
df.info() # data sudah bersih

### Select X and y

In [3]:
X = df.drop(columns = 'churn_value', inplace = False)
y = df['churn_value']

### Naive Bayes: Bernoulli

In [4]:
# Prosedur Bernoulli Naive Bayes

from IPython.display import display as dis, HTML as web
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report
import jinja2

# 1. Membagi data:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% untuk testing (X_test, y_test)
    random_state=42,    
    stratify=y          # memastikan proporsi kelas churn/tidak churn seimbang di kedua data
)

# 2. Inisialisasi model Bernoulli NB
model_nb = BernoulliNB(alpha=1.0) # alpha=1.0 nilai Laplace Smoothing (+1)

# 3. Training: Hitung prior dan conditional prob
model_nb.fit(X_train, y_train)

# 4. Proses prediksi data uji
y_pred = model_nb.predict(X_test)

# skor persentase masing-masing kelas
y_pred_proba = model_nb.predict_proba(X_test)[:, 1]

# 5. Evaluasi
print("\nQuick Report: Data Uji")
print(classification_report(y_test, y_pred))


Quick Report: Data Uji
              precision    recall  f1-score   support

           0       0.73      0.71      0.72       370
           1       0.72      0.73      0.73       370

    accuracy                           0.72       740
   macro avg       0.72      0.72      0.72       740
weighted avg       0.72      0.72      0.72       740



### Implementation of a new customer profile

In [ ]:
new = {
    "senior_citizen" : 0,
    "partner" : 0,
    "dependents" : 1,
    "phone_service" : 1,
    "multiple_lines" : 1,
    "internet_service" : 1,
    "online_security" : 0,
    "tech_support" : 1,
    "streaming_tv" : 0,
    "streaming_movies" : 1
}

new_profile = pd.DataFrame([new])                       # profil baru
prediksi_label = model_nb.predict(new_profile)          # preidiksi label
prediksi_peluang = model_nb.predict_proba(new_profile)  # prediksi peluang proporsional

if prediksi_label[0] == 1:
    print("Pelanggan ini diprediksi CHURN")
else:
    print("Pelanggan ini diprediksi TIDAK CHURN")

print("\nPROBABILITAS")
# indeks 0 = Kelas 0 (Tidak Churn), indeks 1 = Kelas 1 (Churn)
print(f"Peluang TIDAK CHURN : {prediksi_peluang[0][0] * 100:.2f}%")
print(f"Peluang CHURN       : {prediksi_peluang[0][1] * 100:.2f}%")